In [1]:
import subprocess
import sys
import importlib.metadata as metadata

packages = [
    "qiskit==2.2.3", "numpy==2.2.6", "scipy==1.15.3",
    "scikit-learn==1.6.1", "pandas==2.2.3", "matplotlib==3.10.1",
    "flask==3.1.0", "joblib==1.6.0", "jedi"
]

subprocess.check_call([sys.executable, "-m", "pip", "install", *packages])
subprocess.check_call([sys.executable, "-m", "pip", "check"])

print("Python:", sys.version.split()[0])
print({p.split("==")[0]: metadata.version(p.split("==")[0]) for p in packages})

Python: 3.13.18
{'qiskit': '2.2.3', 'numpy': '2.2.6', 'scipy': '1.15.3', 'scikit-learn': '1.6.1', 'pandas': '2.2.3', 'matplotlib': '3.10.1', 'flask': '3.1.0', 'joblib': '1.6.0', 'jedi': '0.20.1'}


In [5]:
import subprocess
import sys
import importlib.metadata as metadata

packages = [
    "qiskit==2.2.3", "numpy==2.2.6", "scipy==1.15.3",
    "scikit-learn==1.6.1", "pandas==2.2.3", "matplotlib==3.10.1",
    "flask==3.1.0", "joblib==1.6.0", "jedi"
]

subprocess.check_call([sys.executable, "-m", "pip", "install", *packages])
subprocess.check_call([sys.executable, "-m", "pip", "check"])

print("Python:", sys.version.split()[0])
print({p.split("==")[0]: metadata.version(p.split("==")[0]) for p in packages})

Python: 3.13.18
{'qiskit': '2.2.3', 'numpy': '2.2.6', 'scipy': '1.15.3', 'scikit-learn': '1.6.1', 'pandas': '2.2.3', 'matplotlib': '3.10.1', 'flask': '3.1.0', 'joblib': '1.6.0', 'jedi': '0.20.1'}


In [6]:
for n in [2,6,20,30,50]:
    state_bytes=16*2**n
    print('qubits:',n,'amplitudes:',2**n,'bytes:',state_bytes,
          'GiB:',state_bytes/2**30)

qubits: 2 amplitudes: 4 bytes: 64 GiB: 5.960464477539063e-08
qubits: 6 amplitudes: 64 bytes: 1024 GiB: 9.5367431640625e-07
qubits: 20 amplitudes: 1048576 bytes: 16777216 GiB: 0.015625
qubits: 30 amplitudes: 1073741824 bytes: 17179869184 GiB: 16.0
qubits: 50 amplitudes: 1125899906842624 bytes: 18014398509481984 GiB: 16777216.0


In [7]:
import numpy as np
import pandas as pd
from io import StringIO
from time import perf_counter
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.decomposition import PCA
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, balanced_accuracy_score, f1_score, confusion_matrix
from qiskit.circuit.library import zz_feature_map, pauli_feature_map
from qiskit.quantum_info import Statevector

# source_row: scikit-learn 1.6.1 WDBC 데이터의 0부터 시작하는 행 번호
# 열 순서: source_row, radius, texture, perimeter, area, smoothness, compactness, label
# label: 0=원자료 양성(benign), 1=원자료 악성(malignant)
raw = """19,13.54,14.36,87.46,566.3,0.09779,0.08129,0
20,13.08,15.71,85.63,520,0.1075,0.127,0
21,9.504,12.44,60.34,273.9,0.1024,0.06492,0
37,13.03,18.42,82.61,523.8,0.08983,0.03766,0
46,8.196,16.84,51.71,201.9,0.086,0.05943,0
48,12.05,14.63,78.04,449.3,0.1031,0.09092,0
49,13.49,22.3,86.91,561,0.08752,0.07698,0
50,11.76,21.6,74.72,427.9,0.08637,0.04966,0
51,13.64,16.34,87.21,571.8,0.07685,0.06059,0
52,11.94,18.24,75.71,437.6,0.08261,0.04751,0
55,11.52,18.75,73.34,409,0.09524,0.05473,0
58,13.05,19.31,82.61,527.2,0.0806,0.03789,0
59,8.618,11.79,54.34,224.5,0.09752,0.05272,0
60,10.17,14.88,64.55,311.9,0.1134,0.08061,0
61,8.598,20.98,54.66,221.8,0.1243,0.08963,0
63,9.173,13.86,59.2,260.9,0.07721,0.08751,0
66,9.465,21.01,60.11,269.4,0.1044,0.07773,0
67,11.31,19.04,71.8,394.1,0.08139,0.04701,0
68,9.029,17.33,58.79,250.5,0.1066,0.1413,0
69,12.78,16.49,81.37,502.5,0.09831,0.05234,0
0,17.99,10.38,122.8,1001,0.1184,0.2776,1
1,20.57,17.77,132.9,1326,0.08474,0.07864,1
2,19.69,21.25,130,1203,0.1096,0.1599,1
3,11.42,20.38,77.58,386.1,0.1425,0.2839,1
4,20.29,14.34,135.1,1297,0.1003,0.1328,1
5,12.45,15.7,82.57,477.1,0.1278,0.17,1
6,18.25,19.98,119.6,1040,0.09463,0.109,1
7,13.71,20.83,90.2,577.9,0.1189,0.1645,1
8,13,21.82,87.5,519.8,0.1273,0.1932,1
9,12.46,24.04,83.97,475.9,0.1186,0.2396,1
10,16.02,23.24,102.7,797.8,0.08206,0.06669,1
11,15.78,17.89,103.6,781,0.0971,0.1292,1
12,19.17,24.8,132.4,1123,0.0974,0.2458,1
13,15.85,23.95,103.7,782.7,0.08401,0.1002,1
14,13.73,22.61,93.6,578.3,0.1131,0.2293,1
15,14.54,27.54,96.73,658.8,0.1139,0.1595,1
16,14.68,20.13,94.74,684.5,0.09867,0.072,1
17,16.13,20.68,108.1,798.8,0.117,0.2022,1
18,19.81,22.15,130,1260,0.09831,0.1027,1
22,15.34,14.26,102.5,704.4,0.1073,0.2135,1"""
df = pd.read_csv(StringIO(raw), header=None,
    names=['source_row','radius','texture','perimeter','area','smoothness','compactness','label'])
FEATURES = list(df.columns[1:7])
X = df[FEATURES].to_numpy(float)
y = df.label.to_numpy(int)
# 각 클래스의 앞 14개 학습, 뒤 6개 평가: 고정 분할
train = np.r_[0:14,20:34]
test = np.r_[14:20,34:40]
assert set(train).isdisjoint(test) and np.isfinite(X).all()
assert np.array_equal(np.bincount(y[train]), [14,14])
assert np.array_equal(np.bincount(y[test]), [6,6])

def prepare(n=2, indices=None):
    indices = train if indices is None else np.asarray(indices)
    standard = StandardScaler().fit(X[indices])
    pca = PCA(n_components=n, svd_solver='full').fit(standard.transform(X[indices]))
    angle = MinMaxScaler(feature_range=(0,1)).fit(pca.transform(standard.transform(X[indices])))
    def transform(values):
        # 학습 범위 밖 값은 0~1 rad 경계로 제한합니다.
        return np.clip(angle.transform(pca.transform(standard.transform(values))),0,1)
    transform.steps=(standard,pca,angle)
    return transform(X[indices]),transform(X[test]),transform,pca

def states(values, kind='ZZ', reps=1):
    n=values.shape[1]
    circuit=(zz_feature_map(n,reps=reps,entanglement='linear') if kind=='ZZ'
        else pauli_feature_map(n,reps=reps,entanglement='linear',paulis=['X','ZZ']))
    parameters=list(circuit.parameters)
    vectors=np.array([Statevector.from_instruction(circuit.assign_parameters(
        dict(zip(parameters,row)))).data for row in values])
    assert np.allclose(np.sum(abs(vectors)**2,axis=1),1)
    return vectors,circuit

def gram(a,b):
    return np.clip(abs(a.conj() @ b.T)**2,0,1)

def evaluate(a,b,labels,kind='ZZ',reps=1,weight=None):
    started=perf_counter()
    sa,circuit=states(a,kind,reps)
    sb,_=states(b,kind,reps)
    k=gram(sa,sa); kt=gram(sb,sa)
    kernel_seconds=perf_counter()-started
    assert np.allclose(k,k.T) and np.allclose(k.diagonal(),1)
    assert np.linalg.eigvalsh(k).min()>-1e-8
    models={'quantum':SVC(kernel='precomputed',C=1,class_weight=weight),
            'rbf':SVC(kernel='rbf',C=1,gamma='scale',class_weight=weight)}
    records=[]; predictions={}
    for name,model in models.items():
        t=perf_counter()
        model.fit(k if name=='quantum' else a,labels)
        pred=model.predict(kt if name=='quantum' else b)
        seconds=perf_counter()-t
        predictions[name]=pred
        records.append(dict(model=name,accuracy=accuracy_score(y[test],pred),
            balanced_accuracy=balanced_accuracy_score(y[test],pred),
            f1=f1_score(y[test],pred,zero_division=0),
            kernel_seconds=kernel_seconds if name=='quantum' else 0,
            fit_predict_seconds=seconds,
            total_seconds=seconds+(kernel_seconds if name=='quantum' else 0)))
        print(name,'confusion matrix [0,1] rows=true cols=predicted')
        print(confusion_matrix(y[test],pred,labels=[0,1]))
    return pd.DataFrame(records),models,sa,k,predictions,circuit

all_results=[]
for n in [2,3,4,6]:
    a,b,_,pca=prepare(n)
    result,*_=evaluate(a,b,y[train])
    result['qubits']=n
    result['retained_variance']=pca.explained_variance_ratio_.sum()
    result['state_bytes']=16*2**n
    all_results.append(result)
summary=pd.concat(all_results,ignore_index=True)
print(summary.round(6).to_string(index=False))
summary.to_csv('pca_results.csv',index=False)

quantum confusion matrix [0,1] rows=true cols=predicted
[[6 0]
 [1 5]]
rbf confusion matrix [0,1] rows=true cols=predicted
[[6 0]
 [1 5]]
quantum confusion matrix [0,1] rows=true cols=predicted
[[5 1]
 [0 6]]
rbf confusion matrix [0,1] rows=true cols=predicted
[[6 0]
 [0 6]]
quantum confusion matrix [0,1] rows=true cols=predicted
[[4 2]
 [1 5]]
rbf confusion matrix [0,1] rows=true cols=predicted
[[5 1]
 [0 6]]
quantum confusion matrix [0,1] rows=true cols=predicted
[[2 4]
 [1 5]]
rbf confusion matrix [0,1] rows=true cols=predicted
[[5 1]
 [0 6]]
  model  accuracy  balanced_accuracy       f1  kernel_seconds  fit_predict_seconds  total_seconds  qubits  retained_variance  state_bytes
quantum  0.916667           0.916667 0.909091        0.086554             0.002747       0.089301       2           0.823058           64
    rbf  0.916667           0.916667 0.909091        0.000000             0.001110       0.001110       2           0.823058           64
quantum  0.916667           0.9166

차원이 바뀌면 표본의 양자 상태와 커널 유사도도 바뀝니다. 6차원 양자 커널 결과의 혼동행렬은 [[2,4],[1,5]]입니다. 분산 보존율 1.0은 여섯 입력 특징의 분산이 보존됨을 나타내며, 예측 정확도는 7/12입니다. 표의 변화는 이 고정 표본과 각도 범위, 한 층 회로의 관측 결과입니다. 소규모 상태 배열의 바이트 수에는 라이브러리와 Python 객체의 메모리 사용량이 포함되지 않습니다.

###